# Task 4 · Step 3 — Manual audit of the AI judge

**Policies (fixed by the manual).** SFT = untouched `Qwen2.5-1.5B-Instruct`; DPO = standard one-epoch adapter (Task 1 Step 1); PPO = standard 20-update continuation (Task 2 Step 1); GRPO = standard 20-update continuation (Task 3 Step 1). No ablation checkpoint is substituted, and Task 4 results were not used to tune Tasks 1–3.

**Protocol.** One deterministic (greedy) response per policy and XSTest prompt (450 prompts: 250 SAFE, 200 UNSAFE contrasts), cap 256 tokens. The fixed course judge (`Qwen2.5-3B-Instruct`, 4-bit, greedy) assigns one label from {SAFE_ANSWER, JUSTIFIED_REFUSAL, UNSAFE_COMPLIANCE, OVER_REFUSAL, AMBIGUOUS}; it sees the prompt and response but not the XSTest class.

**Metrics (released judge spec).** safe-answer = P(SAFE_ANSWER | SAFE), over-refusal = P(OVER_REFUSAL | SAFE), unsafe-compliance = P(UNSAFE_COMPLIANCE | UNSAFE), justified-refusal = P(JUSTIFIED_REFUSAL | UNSAFE), ambiguous = P(AMBIGUOUS). Because the judge does not see the class, it can return a label that is inconsistent with it (e.g. JUSTIFIED_REFUSAL for a refused SAFE prompt); those cells are shown separately, together with the supplementary any-refusal rate on SAFE prompts. All rates carry 95% Wilson intervals.

**Audit design.** The fixed 60-prompt subset (30 SAFE + 30 UNSAFE, sampled with seed 6304, `results/task4_safety/manual_audit_ids.csv`) is joined to each policy's response: 240 responses. The sheet `results/task4_safety/manual_audit_sheet.csv` is blind (shuffled rows, no policy name, no AI label); the mapping is in `manual_audit_key.csv`, which should stay closed while labelling. This notebook never prints which policy produced an audit row before the labels are filled.

**Research question (RQ3).** What kinds of responses does the AI safety judge misclassify, and how much do those errors affect the policy comparison?


In [1]:
%matplotlib inline
import os, sys, json, re
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml
from IPython.display import display, Markdown, HTML, Image
from common.nb_helpers import (style, load, jl, exists, pending, run, side_by_side, ci_text,
                               bootstrap_mean_diff, smooth, full_windows, PALETTE, SEQ, RES, FIG)
from common.metrics import length_stats, wilson_interval
from common.data import load_yaml
style()
print("repo root:", ROOT)

repo root: C:\Users\moize\OneDrive - Higher Education Commission\Desktop\Academics\ATML\PA2


## Run this step (optional)
The results below come from the Kaggle run of the script pipeline (`kaggle_runner.ipynb`). Set `FORCE_RUN = True`
to re-run the step's scripts from here (needs a CUDA GPU and the downloaded course assets). `make_audit_sheet` never overwrites a sheet that already contains labels.


In [2]:
FORCE_RUN = False
if FORCE_RUN:
    run('python -m task4_safety.make_audit_sheet --config configs/feedback.yaml')
    run('python -m task4_safety.evaluate_safety --config configs/feedback.yaml')
else:
    print('Using saved results under results/ (FORCE_RUN = False).')

Using saved results under results/ (FORCE_RUN = False).


## 1. The audited prompts

In [3]:
ids = pd.read_csv("results/task4_safety/manual_audit_ids.csv")
X = pd.read_csv("data/xstest_safety_prompts.csv").set_index("xstest_id")
A = X.loc[ids.xstest_id]
display(pd.crosstab(A["type"], A["benchmark_class"], margins=True))
sheet = pd.read_csv("results/task4_safety/manual_audit_sheet.csv")
lab = sheet.manual_label.fillna("").astype(str).str.strip()
print(f"audit sheet rows: {len(sheet)} (= {len(ids)} prompts × 4 policies); labelled so far: {(lab != '').sum()}")
print("label counts:", lab[lab != ""].value_counts().to_dict())

benchmark_class,SAFE,UNSAFE,All
type,,,
contrast_definitions,0,4,4
contrast_discr,0,2,2
contrast_figurative_language,0,3,3
contrast_historical_events,0,8,8
contrast_homonyms,0,3,3
contrast_privacy,0,2,2
contrast_safe_contexts,0,4,4
contrast_safe_targets,0,4,4
definitions,1,0,1


audit sheet rows: 240 (= 60 prompts × 4 policies); labelled so far: 0
label counts: {}


## 2. Agreement between the manual labels and the AI judge

In [4]:
S = load("task4_safety/safety_evaluation_results.json"); AUD = S.get("_manual_audit")
LABELS = ["SAFE_ANSWER", "JUSTIFIED_REFUSAL", "UNSAFE_COMPLIANCE", "OVER_REFUSAL", "AMBIGUOUS"]
if not AUD:
    display(Markdown("> **Pending:** the audit sheet has no labels yet. Fill `manual_label` in `results/task4_safety/manual_audit_sheet.csv`, run "
                     "`python -m task4_safety.evaluate_safety --config configs/feedback.yaml`, then re-execute this notebook."))
else:
    rows = {p.upper(): v for p, v in AUD["per_policy"].items() if v}; rows["pooled"] = AUD["pooled"]
    display(pd.DataFrame({k: {"n": v["n"], "agreement": v["agreement"], "95% CI": "[{:.2f}, {:.2f}]".format(*v["agreement_ci95"]), "Cohen κ": v["cohen_kappa"],
                              "AI ambiguous": v["ai_ambiguous_rate"], "manual ambiguous": v["manual_ambiguous_rate"],
                              "AI conf. when agree / disagree": f"{v['ai_confidence_when_agree']:.2f} / {v['ai_confidence_when_disagree']:.2f}"} for k, v in rows.items()}).T)
    C = np.array([[AUD["pooled"]["confusion_manual_rows_ai_cols"][m][a] for a in LABELS] for m in LABELS])
    fig, ax = plt.subplots(figsize=(6.5, 5.2)); ax.imshow(C, cmap="Blues"); ax.grid(False)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, C[i, j], ha="center", va="center", color="white" if C[i, j] > C.max() / 2 else "black")
    short = ["SAFE_ANS", "JUST_REF", "UNSAFE_C", "OVER_REF", "AMBIG"]
    ax.set_xticks(range(5)); ax.set_xticklabels(short, rotation=30); ax.set_yticks(range(5)); ax.set_yticklabels(short)
    ax.set_xlabel("AI judge"); ax.set_ylabel("manual"); ax.set_title("pooled confusion (240 responses)"); plt.show()

> **Pending:** the audit sheet has no labels yet. Fill `manual_label` in `results/task4_safety/manual_audit_sheet.csv`, run `python -m task4_safety.evaluate_safety --config configs/feedback.yaml`, then re-execute this notebook.

## 3. Does judge error change the policy comparison? (audited subset: AI labels vs manual labels)

In [5]:
if AUD:
    rs = AUD["audit_subset_rates"]
    display(pd.DataFrame({(p.upper(), src): rs[p][src] for p in rs for src in ["ai", "manual"]}).T.round(3))
else:
    print("pending manual labels")

pending manual labels


## 4. Disagreement examples (judge errors or policy subtleties)

In [6]:
if AUD:
    for d in AUD["disagreements"][:8]:
        side_by_side(f"{d['policy'].upper()} · [{d['benchmark_class']} · {d['type']}] manual = {d['manual']}, AI = {d['ai']}", d["prompt"],
                     {"response": d["response"]}, meta={"response": f"AI confidence {d['ai_confidence']:.2f}; note: {d['notes']}"}, limit=500)
else:
    print("pending manual labels")

pending manual labels
